# 01 - Data Preprocessing

## MNIST Digit Classification: 4 vs 9

This notebook loads the MNIST dataset, filters for digits 4 and 9, and applies preprocessing including normalization, standardization, and PCA dimensionality reduction.

## 1. Introduction to Quantum Machine Learning

### What is Quantum Machine Learning?

Quantum Machine Learning (QML) explores the intersection of quantum computing and machine learning. The fundamental idea is to leverage quantum mechanical properties—such as superposition, entanglement, and interference—to potentially enhance or accelerate machine learning algorithms.

### Key Approaches

1. **Quantum-Enhanced Feature Spaces**: Map classical data into high-dimensional quantum Hilbert spaces
2. **Variational Quantum Circuits**: Use parameterized quantum circuits as function approximators
3. **Quantum Sampling**: Leverage quantum computers for sampling from complex distributions
4. **Quantum Kernel Methods**: Use quantum computers to compute kernel matrices

### This Project

This project focuses on **Quantum Kernel Methods**, specifically comparing classical RBF kernel SVM against Quantum Kernel SVM.

## 2. Setup and Imports

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import sys
import os

# Add src to path
sys.path.append(os.path.join(os.path.dirname('../'), 'src'))

# Set random seed for reproducibility
np.random.seed(42)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 3. Load MNIST Dataset

### Dataset Selection: Digits 4 vs 9

We select digits **4 and 9** because they have similar shapes, making this a challenging binary classification problem. This provides a good testbed for comparing classical and quantum approaches.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Load MNIST dataset
print("Loading MNIST dataset from OpenML...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_full = mnist.data
y_full = mnist.target.astype(int)

print(f"Total MNIST samples: {len(X_full)}")
print(f"Features per sample: {X_full.shape[1]}")
print(f"Unique labels: {np.unique(y_full)}")

In [ ]:
# Filter for digits 4 and 9
digits = [4, 9]
mask = np.isin(y_full, digits)
X = X_full[mask]
y = y_full[mask]

print(f"\nFiltered dataset (digits {digits}):")
print(f"Total samples: {len(X)}")
print(f"Class distribution:")
for digit in digits:
    count = np.sum(y == digit)
    print(f"  Digit {digit}: {count} samples ({100*count/len(y):.1f}%)")

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining class distribution:")
for digit in digits:
    count = np.sum(y_train == digit)
    print(f"  Digit {digit}: {count}")
print(f"\nTest class distribution:")
for digit in digits:
    count = np.sum(y_test == digit)
    print(f"  Digit {digit}: {count}")

## 4. Visualize Sample Images

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, digit in enumerate(digits):
    digit_indices = np.where(y_train == digit)[0][:5]
    for j, idx in enumerate(digit_indices):
        img = X_train[idx].reshape(28, 28)
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(f'Digit {digit}', fontsize=12)

plt.suptitle('Sample MNIST Images: Digits 4 vs 9', fontsize=14)
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample images saved to results/sample_images.png")

## 5. Preprocessing Pipeline

### 5.1 Normalization

First, normalize pixel values to [0, 1] range by dividing by 255.

In [ ]:
# Step 1: Normalize pixel values to [0, 1]
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

print("Step 1: Normalization")
print(f"  Training data range: [{X_train_norm.min():.4f}, {X_train_norm.max():.4f}]")
print(f"  Mean: {X_train_norm.mean():.4f}")
print(f"  Std: {X_train_norm.std():.4f}")

### 5.2 Standardization

Standardize features to have zero mean and unit variance using StandardScaler.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Step 2: Standardize features
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_norm)
X_test_std = scaler.transform(X_test_norm)

print("Step 2: Standardization")
print(f"  Training data range: [{X_train_std.min():.4f}, {X_train_std.max():.4f}]")
print(f"  Mean: {X_train_std.mean():.4f}")
print(f"  Std: {X_train_std.std():.4f}")

### 5.3 PCA Dimensionality Reduction

**Why PCA?**

Quantum computers have limited qubits (NISQ era: 50-100+ qubits). MNIST has 784 features, which would require too many qubits. PCA reduces the feature space to 4 dimensions (matching 4 qubits) while preserving most variance.

> **Note**: PCA was used to preserve 95% of the variance while reducing the feature set to 4 dimensions to accommodate the qubit constraints of NISQ-era hardware.

In [ ]:
from sklearn.decomposition import PCA

# Step 3: Apply PCA for dimensionality reduction
n_components = 4
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

print("Step 3: PCA Dimensionality Reduction")
print(f"  Original dimensions: {X_train_std.shape[1]}")
print(f"  Reduced dimensions: {X_train_pca.shape[1]}")
print(f"\n  Variance explained per component:")
for i, var in enumerate(pca.explained_variance_ratio_):
    cumulative = np.sum(pca.explained_variance_ratio_[:i+1])
    print(f"    PC{i+1}: {var:.4f} (cumulative: {cumulative:.4f})")

### 5.4 Quantum Scaling

Scale features to [0, π] range for quantum angle encoding. Many quantum feature maps encode data through rotation angles.

In [ ]:
# Step 4: Scale for quantum encoding [0, pi]
X_train_quantum = np.pi * (X_train_pca - X_train_pca.min(axis=0)) / (X_train_pca.max(axis=0) - X_train_pca.min(axis=0) + 1e-12)
X_test_quantum = np.pi * (X_test_pca - X_train_pca.min(axis=0)) / (X_train_pca.max(axis=0) - X_train_pca.min(axis=0) + 1e-12)

print("Step 4: Quantum Scaling [0, π]")
print(f"  Training data range: [{X_train_quantum.min():.4f}, {X_train_quantum.max():.4f}]")
print(f"  Mean: {X_train_quantum.mean():.4f}")

## 6. PCA Visualization

### 6.1 PCA Scatter Plot

In [ ]:
# Plot PCA scatter (PC1 vs PC2)
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#3498db', '#e74c3c']
markers = ['o', 's']

for i, digit in enumerate(digits):
    mask = y_train == digit
    ax.scatter(
        X_train_pca[mask, 0], 
        X_train_pca[mask, 1],
        c=colors[i],
        marker=markers[i],
        label=f'Digit {digit}',
        alpha=0.6,
        edgecolors='black',
        s=50
    )

ax.set_xlabel('Principal Component 1', fontsize=12)
ax.set_ylabel('Principal Component 2', fontsize=12)
ax.set_title('PCA Scatter Plot: MNIST Digits 4 vs 9', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("PCA scatter plot saved to results/pca_scatter.png")

### 6.2 PCA Variance Explained

In [ ]:
# Plot PCA variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

# Individual variance
axes[0].bar(range(1, n_components + 1), variance_ratio, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Individual Variance Explained per Component')
axes[0].set_xticks(range(1, n_components + 1))

# Cumulative variance
axes[1].plot(range(1, n_components + 1), cumulative_variance, 'o-', color='steelblue', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
axes[1].fill_between(range(1, n_components + 1), cumulative_variance, alpha=0.3, color='steelblue')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Variance Explained')
axes[1].set_xticks(range(1, n_components + 1))
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print("PCA variance plot saved to results/pca_variance.png")

### 6.3 Class Distribution

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(8, 6))

counts = [np.sum(y_train == digit) for digit in digits]
bars = ax.bar([str(d) for d in digits], counts, color=colors, edgecolor='black')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 20, 
            str(count), ha='center', va='bottom', fontsize=12)

ax.set_xlabel('Digit Class')
ax.set_ylabel('Number of Samples')
ax.set_title('Training Set Class Distribution')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Class distribution plot saved to results/class_distribution.png")

## 7. Save Preprocessed Data

In [ ]:
# Save preprocessed data for later use
np.save('../results/X_train_pca.npy', X_train_pca)
np.save('../results/X_test_pca.npy', X_test_pca)
np.save('../results/X_train_quantum.npy', X_train_quantum)
np.save('../results/X_test_quantum.npy', X_test_quantum)
np.save('../results/y_train.npy', y_train)
np.save('../results/y_test.npy', y_test)

print("Preprocessed data saved!")
print(f"  X_train_pca: {X_train_pca.shape}")
print(f"  X_test_pca: {X_test_pca.shape}")
print(f"  X_train_quantum: {X_train_quantum.shape}")
print(f"  X_test_quantum: {X_test_quantum.shape}")

## 8. Summary

In [ ]:
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print(f"\nDataset: MNIST Digits {digits}")
print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nPCA Components: {n_components}")
print(f"Variance Explained: {np.sum(pca.explained_variance_ratio_):.4f}")
print(f"\nFeature dimensions after PCA: {X_train_pca.shape[1]}")
print(f"Quantum encoding range: [0, π]")
print("\n" + "=" * 60)

---

## Next Steps

Proceed to **02_classical_baseline.ipynb** to train and evaluate the classical RBF SVM model.